# Phase 7 · one-factor controls
Six runs for the report: loss, negatives, pooling and Phase 1 transfer. Dev only. Outputs go to `runs/phase7_pcr/p7x-*/` on Drive; finished runs are skipped; the last cell disconnects the runtime.

In [ ]:
# 1 · Settings
from pathlib import Path
SEED = 17
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
GQA, ENC = 'pretrain/pretrain_dec_gqa/model_final.pt', 'pretrain/pretrain_enc_mlm/model_final.pt'
Q_BUDGET, C_BUDGET = 512, 512      # excerpt tokens per document; a pair is about 1026 tokens
TRAIN_TOP, EVAL_TOP = 40, 50       # lexical shortlist: hard negatives for training, candidates reranked on dev / test
BASE = dict(epochs=2, accum=4, negs=7, pos_per_q=3, lr_head=1e-4, seed=SEED, max_minutes=40, log_every=200, eval_batch=32, kd_t=2.0, kd_alpha=0.5)
ADAPTER = dict(rank=16, alpha=32, targets=('query', 'value'))
THIN = (0, 2, 4, 6, 8, 10, 12, 14)
RUNS = [  # run id, kind, method, pool, checkpoint, overrides
    ('p7x-enc-pairwise', 'encoder', 'full', 'mean', ENC, dict(lr=2e-5, loss='pairwise')),                # loss: pairwise logistic, baseline listwise softmax
    ('p7x-enc-randneg', 'encoder', 'full', 'mean', ENC, dict(lr=2e-5, negatives='random')),              # negatives: random pool cases, baseline lexical top 40
    ('p7x-enc-last', 'encoder', 'full', 'last', ENC, dict(lr=2e-5)),                                     # pooling: last token, baseline mean
    ('p7x-gqa-mean', 'gqa', 'full', 'mean', GQA, dict(lr=2e-5)),                                         # pooling: mean, baseline last token (p7-gqa-full)
    ('p7x-enc-frozen', 'encoder', 'partial', 'mean', ENC, dict(lr=2e-5, lr_head=1e-3)),                  # transfer: no block trained, head only
    ('p7x-enc-random', 'encoder', 'full', 'mean', ENC, dict(lr=1e-4, randomize=True)),                   # transfer: no Phase 1 weights; higher lr gives scratch a fair chance
]

In [ ]:
# 2 · Setup: Drive, libraries from legal-llm-course/phase7_pcr, tokenizer
from google.colab import drive; drive.mount('/content/drive')
import sys, shutil, zipfile, json, random, time, hashlib
WORK = Path('/content/work'); (WORK/'lib').mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(COURSE_ROOT/'phase2_lner_multimethod_bundle.zip') as z: z.extractall(WORK/'lib')     # legal_scaffold, training_methods, tokenizer
for name in ('rr_lib.py', 'encoder.py', 'hybrid.py', 'pcr_lib.py', 'rerank_lib.py'): shutil.copy(COURSE_ROOT/'phase7_pcr'/name, WORK/'lib'/name)
sys.path[:0] = [str(WORK/'lib')]
import numpy as np, torch
from tokenizers import Tokenizer
import pcr_lib as P, rerank_lib as K
TOKENIZER = WORK/'lib/tokenizer/tokenizer.json'
assert hashlib.sha256(TOKENIZER.read_bytes()).hexdigest() == '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab', 'tokenizer changed'
TOK = Tokenizer.from_file(str(TOKENIZER))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_DIR = COURSE_ROOT/'runs/phase7_pcr'; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

In [ ]:
# 3 · Pairs: lexical shortlists and excerpts for train / dev / test, cached on Drive
ROOT = COURSE_ROOT/'Phase_7_PCR'; CACHE = ROOT/f'rerank_pairs_q{Q_BUDGET}_c{C_BUDGET}_t{TRAIN_TOP}_e{EVAL_TOP}.pt'
if CACHE.is_file():
    DATA = torch.load(CACHE, weights_only=False)
else:
    DATA = dict(seqs={}, train=[]); t0 = time.time()
    for split, top in (('train', TRAIN_TOP), ('dev', EVAL_TOP), ('test', EVAL_TOP)):
        q, c = P.load_split(ROOT, split); qi, ranked, lex = K.shortlist(q, c, top)
        docs = K.prepare_docs(K.load_sentences(ROOT, split), TOK); pairs = []
        if split == 'train':
            for k in qi:
                gold = [g for g in (q[k]['gold'] or []) if g in c and g != k]; neg = [x for x in ranked[k] if x not in set(gold)]
                DATA['train'].append((k, gold, neg)); pairs += [(k, x) for x in gold + neg]
        else:
            pairs = [(k, x) for k in qi for x in ranked[k]]
            DATA[f'{split}_q'], DATA[f'{split}_rank'], DATA[f'{split}_lex'], DATA[f'{split}_pairs'] = qi, ranked, lex, pairs
            if split == 'dev': DATA['dev_gold'] = {k: q[k]['gold'] or [] for k in qi}
            else: DATA['test_pool'] = list(c)
        for p in pairs:
            if p not in DATA['seqs']: DATA['seqs'][p] = K.pair_ids(docs, p[0], p[1], Q_BUDGET, C_BUDGET)
        print(f'{split}: {len(qi)} queries, {len(pairs)} pairs, {(time.time() - t0) / 60:.1f} min'); del docs
    torch.save(DATA, CACHE); print('saved', CACHE)
print(len(DATA['seqs']), 'pairs | mean tokens', int(np.mean([len(v) for v in DATA['seqs'].values()])))
lex_dev = P.micro_prf({k: v[:7] for k, v in DATA['dev_rank'].items()}, DATA['dev_gold']); print('lexical dev', lex_dev)

In [ ]:
# 4 · Random negatives: same number per query as the lexical list, from the whole train pool, gold and self excluded
q, c = P.load_split(ROOT, 'train'); docs = K.prepare_docs(K.load_sentences(ROOT, 'train'), TOK); rng = random.Random(SEED); pool = sorted(c); RAND = []; t0 = time.time()
for k, gold, neg in DATA['train']:
    bad = set(gold) | {k}; r = rng.sample([x for x in pool if x not in bad], len(neg)); RAND.append((k, gold, r))
    for x in r:
        if (k, x) not in DATA['seqs']: DATA['seqs'][k, x] = K.pair_ids(docs, k, x, Q_BUDGET, C_BUDGET)
del docs; print('random negatives', sum(len(t[2]) for t in RAND), f'{(time.time() - t0) / 60:.1f} min')

In [ ]:
# 5 · Train and score dev. A run with results.json on Drive is skipped
AMP = torch.bfloat16
for run in RUNS:
    rid, kind, method, pool, ckpt, o = run; d = RUN_DIR/rid
    if (d/'results.json').is_file(): print('skip', rid); continue
    print('=' * 20, rid, method, kind, pool)
    C = dict(BASE, run_id=rid, kind=kind, method=method, pool=pool, **{k: v for k, v in o.items() if k not in ('randomize', 'negatives')})
    torch.manual_seed(SEED); model = K.build(COURSE_ROOT/ckpt, kind, method, pool, Q_BUDGET + C_BUDGET + 2, adapter_kw=ADAPTER)
    if o.get('randomize'): K.randomize(model)
    res = K.train_run(model, dict(DATA, train=RAND) if o.get('negatives') == 'random' else DATA, d, C, DEVICE)
    best = K.load_best(model, d); model.to(DEVICE)
    dev = K.score_pairs(model, DATA['dev_pairs'], DATA['seqs'], DEVICE, AMP, BASE['eval_batch']); K.save_scores(d/'dev_scores.json', dev)
    (d/'results.json').write_text(json.dumps(dict(res, lam=best['lam'], table=best['table']), indent=1, default=float)); print(rid, 'done', res['history'])
    del model; torch.cuda.empty_cache()

In [ ]:
# 6 · Release the GPU (saves compute units). Only reached when every cell above succeeded
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed; disconnecting the runtime')
runtime.unassign()